# eNTERFACE’05 Preprocessing Script
This script is used for generating eNTERFACE’05_RAW_PREPROCESSED data from the raw RAW data, you can download the raw dataset from: http://www.enterface.net/results/

emotion: (anger、disgust、fear、happiness、sadness、surprise)

In [1]:
import torch
from facenet_pytorch import MTCNN
import os, sys
import glob
import pickle
import numpy as np
import pandas as pd
import cv2
from scipy.io import wavfile
from tqdm import tqdm

In [2]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(image_size=48, margin=2, post_process=False, device=device)

# Common Functions

In [3]:
def read_video(file_name):
    vidcap = cv2.VideoCapture(file_name)
    
    # Read FPS
    (major_ver, minor_ver, subminor_ver) = (cv2.__version__).split('.')
    if int(major_ver)  < 3 :
        fps = vidcap.get(cv2.cv.CV_CAP_PROP_FPS)
    else :
        fps = vidcap.get(cv2.CAP_PROP_FPS)
    
    # Read image data
    success, image = vidcap.read()
    images = []
    while success:
        images.append(image)
        success, image = vidcap.read()
    return np.stack(images), fps

def dump_image(img_segment, out_path='./'):
    count = 0
    for i in range(img_segment.shape[0]):
        faces = mtcnn(img_segment[i,:,:,:])
        if faces != None:
#             cv2.imwrite(f'{out_path}/image_{i}.jpg', img_segment[i,:,:,:])
            cv2.imwrite(f'{out_path}/image_{count}.jpg', faces.permute(1, 2, 0).int().numpy())
            count = count + 1
#             mtcnn(img_segment[i,:,:,:], save_path=f'{out_path}/image_{i}.jpg')

In [5]:
%%time
# Process multimodal data over all sessions
# NOTE: This might take several hours to run, the time listed on this cell is for processing 5 label files
output_path = r'/home/matt/Model/Database_processed/eNTERFACE’05_RAW_PROCESSED_Face'
#     list wav file
wav_path = r'/home/matt/Model/Data/eNTERFACE’05/audio_file'
wav_list = os.listdir(wav_path)
for i in range(len(wav_list)):
    wav_list[i] = wav_list[i][:-4]
if not os.path.exists(output_path):
    os.makedirs(output_path)
    
all_metas = {}
for base_path in glob.glob(r'/home/matt/Model/Data/eNTERFACE’05/enterface database/*'):
    for root, dirs, files in os.walk(base_path, topdown=False):
        for name in files:
            if name[-3:] == 'avi':
                print(name)
#                 print('1', os.path.join(root, name))
                audio_file_name = name[:-4]
#                 print('2', audio_file_name)
                index = wav_list.index(audio_file_name)
                sr, signal  = wavfile.read(os.path.join(wav_path, wav_list[index]) + '.wav')
                images, fps = read_video(os.path.join(root, name))
#                 print('3', os.path.join(wav_path, wav_list[index]) + '.wav')
                out_path = os.path.join(output_path, audio_file_name)
    #             print(out_path)
                if not os.path.exists(out_path):
                    os.makedirs(out_path)
                wavfile.write(f'{out_path}/audio.wav', sr, signal)
                dump_image(images, out_path)

s8_su_1.avi
s8_su_2.avi
s8_su_5.avi
s8_su_4.avi
s8_su_3.avi
s8_di_1.avi
s8_di_2.avi
s8_di_5.avi
s8_di_4.avi
s8_di_3.avi
s8_fe_1.avi
s8_fe_2.avi
s8_fe_5.avi
s8_fe_4.avi
s8_fe_3.avi
s8_ha_1.avi
s8_ha_2.avi
s8_ha_5.avi
s8_ha_4.avi
s8_ha_3.avi
s8_an_1.avi
s8_an_2.avi
s8_an_5.avi
s8_an_4.avi
s8_an_3.avi
s8_sa_1.avi
s8_sa_2.avi
s8_sa_5.avi
s8_sa_4.avi
s8_sa_3.avi
s36_su_1.avi
s36_su_2.avi
s36_su_5.avi
s36_su_4.avi
s36_su_3.avi
s36_di_1.avi
s36_di_2.avi
s36_di_5.avi
s36_di_4.avi
s36_di_3.avi
s36_fe_1.avi
s36_fe_2.avi
s36_fe_5.avi
s36_fe_4.avi
s36_fe_3.avi
s36_ha_1.avi
s36_ha_2.avi
s36_ha_5.avi
s36_ha_4.avi
s36_ha_3.avi
s36_an_1.avi
s36_an_2.avi
s36_an_5.avi
s36_an_4.avi
s36_an_3.avi
s36_sa_1.avi
s36_sa_2.avi
s36_sa_5.avi
s36_sa_4.avi
s36_sa_3.avi
s31_su_1.avi
s31_su_2.avi
s31_su_5.avi
s31_su_4.avi
s31_su_3.avi
s31_di_1.avi
s31_di_2.avi
s31_di_5.avi
s31_di_4.avi
s31_di_3.avi
s31_fe_1.avi
s31_fe_2.avi
s31_fe_5.avi
s31_fe_4.avi
s31_fe_3.avi
s31_ha_1.avi
s31_ha_2.avi
s31_ha_5.avi
s31_ha_4.avi
s31

In [6]:
df = pd.read_csv('/home/matt/Model/Data/eNTERFACE’05/text.csv')
df

,filename,text
0,s1_an_1.avi,"What? No, no, no, listen, I need this money!"
1,s1_an_2.avi,"I don't care about your coffee, please serve me!"
2,s1_an_3.avi,I can have you fired now!
3,s1_an_4.avi,Is your coffee more important than my money?
4,s1_an_5.avi,"You're getting paid to work, not to drink cof..."
...,...,...
1288,s9_su_1.avi,You never have told me.
1289,s9_su_2.avi,I didn't expect that.
1290,s9_su_3.avi,"Wow, I never would have believed this."
1291,s9_su_4.avi,... a attracted...


In [7]:
metadata = {}
emo_trans = {'ha': 'hap', 'sa': 'sad', 'an': 'ang', 'fe': 'fea', 'su': 'sur', 'di': 'dis'}
df = pd.read_csv(r'/home/matt/Model/Data/eNTERFACE’05/text.csv')
for index, row in df.iterrows():
    print(row['filename'])
    print(row['text'])
    if row['filename'].split('_')[-2] == 's6':
        emo = row['filename'].split('_')[-1][:2]
    else:
        emo = row['filename'].split('_')[-2]
    metadata[row['filename'][:-4]] = {'text': row['text'], 'label': emo_trans[emo]}

pickle.dump(metadata, open(r'/home/matt/Model/Database_processed/eNTERFACE’05_RAW_PROCESSED_Face/meta.pkl','wb'))

s1_an_1.avi
 What? No, no, no, listen, I need this money!
s1_an_2.avi
 I don't care about your coffee, please serve me!
s1_an_3.avi
 I can have you fired now!
s1_an_4.avi
 Is your coffee more important than my money?
s1_an_5.avi
 You're getting paid to work, not to drink coffee!
s1_di_1.avi
 Oh, that's horrible! I never eat noodle again!
s1_di_2.avi
 Oh, something is moving inside my plate.
s1_di_3.avi
 Oh, a cockroach!
s1_di_4.avi
 This is disgusting.
s1_di_5.avi
 That's gross!
s1_fe_1.avi
 Oh my god, there's someone in the house!
s1_fe_2.avi
 Someone is climbing up the stairs.
s1_fe_3.avi
 Please don't kill me.
s1_fe_4.avi
 I'm not alone, go away!
s1_fe_5.avi
 I have nothing to give you. Please don't hurt me.
s1_ha_1.avi
 That's great! I'm rich now!
s1_ha_2.avi
 I won! This is great! I am so happy!
s1_ha_3.avi
 Wow, this is so great!
s1_ha_4.avi
 I'm so lucky!
s1_ha_5.avi
 I'm so excited!
s1_sa_1.avi
 Life can be the same now.
s1_sa_2.avi
 Oh no, please tell me this is not true, plea

In [8]:
import pickle
with open('/home/matt/Model/Database_processed/eNTERFACE’05_RAW_PROCESSED_Face/meta.pkl', 'rb') as f:
    data = pickle.load(f)

In [9]:
data

{'s1_an_1': {'text': ' What? No, no, no, listen, I need this money!',
  'label': 'ang'},
 's1_an_2': {'text': " I don't care about your coffee, please serve me!",
  'label': 'ang'},
 's1_an_3': {'text': ' I can have you fired now!', 'label': 'ang'},
 's1_an_4': {'text': ' Is your coffee more important than my money?',
  'label': 'ang'},
 's1_an_5': {'text': " You're getting paid to work, not to drink coffee!",
  'label': 'ang'},
 's1_di_1': {'text': " Oh, that's horrible! I never eat noodle again!",
  'label': 'dis'},
 's1_di_2': {'text': ' Oh, something is moving inside my plate.',
  'label': 'dis'},
 's1_di_3': {'text': ' Oh, a cockroach!', 'label': 'dis'},
 's1_di_4': {'text': ' This is disgusting.', 'label': 'dis'},
 's1_di_5': {'text': " That's gross!", 'label': 'dis'},
 's1_fe_1': {'text': " Oh my god, there's someone in the house!",
  'label': 'fea'},
 's1_fe_2': {'text': ' Someone is climbing up the stairs.', 'label': 'fea'},
 's1_fe_3': {'text': " Please don't kill me.", 'labe